# Clockwork T4 benchmark

Produces every performance number for the clockwork repo on a free Colab T4 GPU
(Runtime > Change runtime type > T4 GPU). Run top to bottom; each section states a rough
wall-clock range. Nothing is precomputed: every number printed comes from a command
executed in this run. Outputs land in results/, docs/figures/, and a results zip under
/content. Expect 3 to 6 hours total on the free tier.

## 1. GPU check

Confirms a CUDA device is visible via nvidia-smi and torch.cuda, and names it. Every
results table must carry this GPU name; the cell warns if the device is not a T4.
Expected wall clock: under 1 minute.

In [ ]:
import csv
import json
import os
import re
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch


def sh(cmd, echo=True):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        lines.append(line.rstrip("\n"))
        if echo:
            print(line, end="")
    proc.wait()
    return proc.returncode, lines


def must(cmd):
    rc, _ = sh(cmd)
    assert rc == 0, f"exit code {rc}: {' '.join(cmd)}"


try:
    rc, _ = sh(["nvidia-smi"])
except FileNotFoundError:
    rc = 1
assert rc == 0, "nvidia-smi failed: switch to a GPU runtime"
assert torch.cuda.is_available(), "torch sees no CUDA device: switch to a GPU runtime"
GPU_NAME = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
print(f"gpu: {GPU_NAME}")
print(f"memory: {gpu_props.total_memory / 1e9:.1f} GB")
print(f"capability: sm{gpu_props.major}{gpu_props.minor}")
if "T4" not in GPU_NAME:
    print(f"warning: this is not a T4; label every reported number with {GPU_NAME!r}")

## 2. Setup

Clones the repo, installs it editable with the gpu extra plus pytest, pytest-asyncio, and
pynvml (the bench runner samples GPU utilization through pynvml when present), points
HF_HOME under /content, and downloads Qwen/Qwen2.5-1.5B-Instruct. Expected wall clock:
5 to 15 minutes, mostly the model download.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = BASE / "clockwork"
LOGS = BASE / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(BASE / "hf_home")
if not (REPO / "pyproject.toml").exists():
    must(["git", "clone", "https://github.com/jasonjesuraja06/clockwork", str(REPO)])
os.chdir(REPO)
must(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu]"]
    + ["pytest", "pytest-asyncio", "pynvml"]
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

In [ ]:
import transformers
from huggingface_hub import snapshot_download

print("model snapshot:", snapshot_download(MODEL))
print("torch", torch.__version__, "| transformers", transformers.__version__)

## 3. Correctness gates

All gates must pass before any benchmark cell runs, in this order: the slow Hugging Face
exact-match gate on the real model, a preflight proving Triton imports on CUDA (so the
gpu marker cannot skip silently), the gpu-marked Triton kernel tests, and the full
not-slow suite. A failed gate raises and stops the run. Note: the slow gate holds one
float32 copy of the 1.5B model in host RAM at a time (the Hugging Face reference is
freed before the engine loads), which fits the free tier. Expected wall clock:
20 to 60 minutes, dominated by the slow gate on Colab's CPU.

In [ ]:
gates = [
    (
        "hf exact match slow gate",
        [sys.executable, "-m", "pytest", "tests/test_hf_equivalence.py", "-q", "-m", "slow"],
    ),
    (
        "triton usable on this runtime",
        [
            sys.executable,
            "-c",
            "import torch; from clockwork.kernels.triton_paged_attn import HAS_TRITON; "
            "assert HAS_TRITON and torch.cuda.is_available()",
        ],
    ),
    (
        "triton kernel tests, gpu marker",
        [sys.executable, "-m", "pytest", "tests/test_triton_kernel.py", "-q", "-m", "gpu"],
    ),
    (
        "full suite, not slow",
        [sys.executable, "-m", "pytest", "-q", "-m", "not slow"],
    ),
]
GATE_RESULTS = []
for label, cmd in gates:
    t0 = time.monotonic()
    rc, _ = sh(cmd)
    took = time.monotonic() - t0
    assert rc == 0, f"gate failed: {label} (exit {rc})"
    GATE_RESULTS.append((label, " ".join(cmd), took))
    print(f"gate passed: {label} ({took:.0f}s)")
print("all correctness gates passed; benchmarks may run")

## 4. Paged decode kernel microbench

Times the Triton paged decode kernel against the torch fallback on identical tensors at
the served model's shapes (12 query heads, 2 kv heads, head_dim 128, block_size 16 from
the shipped config), batch 1 to 64, context 128 to 2048, float16. CUDA event timing with
warmup and torch.cuda.synchronize, plus a numerical cross-check per shape. The winner is
passed to the server in section 5, matching the resolve_backend policy of keeping the
faster backend. Expected wall clock: 1 to 3 minutes.

In [ ]:
from clockwork.kernels.attention import paged_attention_decode_torch, resolve_backend
from clockwork.kernels.triton_paged_attn import HAS_TRITON, triton_paged_attention_decode

assert HAS_TRITON, "triton did not import; rerun the setup cell"
NUM_HEADS, NUM_KV_HEADS, HEAD_DIM, BLOCK_SIZE = 12, 2, 128, 16
SCALE = HEAD_DIM**-0.5


def make_case(batch, ctx_len, dtype=torch.float16):
    blocks_per_seq = -(-ctx_len // BLOCK_SIZE)
    num_blocks = batch * blocks_per_seq
    shape = (num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_DIM)
    k_cache = torch.randn(shape, device="cuda", dtype=dtype)
    v_cache = torch.randn(shape, device="cuda", dtype=dtype)
    tables = torch.randperm(num_blocks, device="cuda").to(torch.int32)
    tables = tables.reshape(batch, blocks_per_seq)
    ctx_lens = torch.full((batch,), ctx_len, dtype=torch.int32, device="cuda")
    q = torch.randn(batch, NUM_HEADS, HEAD_DIM, device="cuda", dtype=dtype)
    return q, k_cache, v_cache, tables, ctx_lens


def time_ms(fn, args, warmup=10, iters=50):
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn(*args)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters


torch.manual_seed(0)
MICRO_ROWS = []
for batch in (1, 4, 16, 64):
    for ctx_len in (128, 512, 2048):
        case = (*make_case(batch, ctx_len), SCALE)
        ref = paged_attention_decode_torch(*case)
        out = triton_paged_attention_decode(*case)
        torch.testing.assert_close(out, ref, atol=2e-3, rtol=2e-3)
        MICRO_ROWS.append(
            {
                "batch": batch,
                "ctx_len": ctx_len,
                "torch_ms": round(time_ms(paged_attention_decode_torch, case), 4),
                "triton_ms": round(time_ms(triton_paged_attention_decode, case), 4),
            }
        )
        del case, ref, out
torch.cuda.empty_cache()
for row in MICRO_ROWS:
    row["speedup"] = round(row["torch_ms"] / row["triton_ms"], 3)
print(f"{'batch':>6} {'ctx':>6} {'torch_ms':>10} {'triton_ms':>10} {'speedup':>8}")
for row in MICRO_ROWS:
    print(
        f"{row['batch']:>6} {row['ctx_len']:>6} {row['torch_ms']:>10.4f} "
        f"{row['triton_ms']:>10.4f} {row['speedup']:>8.3f}"
    )
total_torch = sum(row["torch_ms"] for row in MICRO_ROWS)
total_triton = sum(row["triton_ms"] for row in MICRO_ROWS)
BACKEND_WINNER = "triton" if total_triton <= total_torch else "torch"
print(
    f"decode winner on {GPU_NAME}: {BACKEND_WINNER} "
    f"(summed mean latency {total_triton:.3f} ms triton vs {total_torch:.3f} ms torch)"
)
print("resolve_backend('auto') here:", resolve_backend("auto"))
(REPO / "results").mkdir(exist_ok=True)
with (REPO / "results" / "microbench_decode.csv").open("w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(MICRO_ROWS[0]))
    writer.writeheader()
    writer.writerows(MICRO_ROWS)

## 5. Clockwork benchmark

Launches scripts/serve.py as a subprocess in float16 (the T4 is sm75, which has no usable
bfloat16), waits for /health, prints /stats to show the attention backend actually in
use, then drives scripts/run_bench.py. The KV cache num_blocks arithmetic for the 16 GB
card is justified in the code comment. The radix-off ablation workloads are replayed
against a second server started with the prefix cache disabled, so every workload row in
results/clockwork/summary.csv is measured under the setting its name claims. Expected
wall clock: 40 to 120 minutes.

In [ ]:
def wait_health(url, proc, timeout_s):
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"server exited early with code {proc.returncode}")
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                if resp.status == 200:
                    return
        except Exception:
            time.sleep(2.0)
    raise TimeoutError(f"{url} not healthy within {timeout_s}s")


def start_server(cmd, log_path, health_url, timeout_s):
    fd = os.open(str(log_path), os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o644)
    proc = subprocess.Popen(cmd, stdout=fd, stderr=subprocess.STDOUT)
    os.close(fd)
    print("launched:", " ".join(cmd))
    try:
        wait_health(health_url, proc, timeout_s)
    except Exception:
        print(Path(log_path).read_text(errors="replace")[-4000:])
        if proc.poll() is None:
            proc.kill()
        raise
    return proc


def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=30)


# One KV block holds k and v (2 tensors) x 28 layers x block_size 16 x 2 kv heads
# x head_dim 128 x 2 bytes fp16 = 458752 bytes. The scheduler ceiling is
# max_num_seqs 64 x max_model_len 4096 = 262144 tokens = 16384 blocks (7.5 GB);
# 2048 extra blocks keep radix prefixes resident after release, so num_blocks
# 18432 = 8.5 GB. With the fp16 weights (about 3.1 GB) plus CUDA context and
# activations, that fits a 16 GB T4 with headroom.
BYTES_PER_BLOCK = 2 * 28 * BLOCK_SIZE * NUM_KV_HEADS * HEAD_DIM * 2
assert BYTES_PER_BLOCK == 458752
NUM_BLOCKS = 18432
kv_gb = NUM_BLOCKS * BYTES_PER_BLOCK / 1e9
print(f"kv cache: {NUM_BLOCKS} blocks x {BYTES_PER_BLOCK} B per block = {kv_gb:.2f} GB")
CLOCKWORK_URL = "http://127.0.0.1:8000"
clockwork_cmd = [
    sys.executable,
    "scripts/serve.py",
    "--config",
    "configs/qwen2.5-1.5b-instruct.yaml",
    "--port",
    "8000",
    "--set",
    "device=cuda",
    "--set",
    "dtype=float16",
    "--set",
    f"num_blocks={NUM_BLOCKS}",
    "--set",
    f"attention_backend={BACKEND_WINNER}",
]
clockwork_proc = start_server(
    clockwork_cmd, LOGS / "clockwork_server.log", f"{CLOCKWORK_URL}/health", timeout_s=900
)
with urllib.request.urlopen(f"{CLOCKWORK_URL}/stats", timeout=10) as resp:
    print(resp.read().decode("utf-8"))

In [ ]:
from clockwork.bench.configs import WORKLOADS

radix_on_names = ",".join(cfg.name for cfg in WORKLOADS if cfg.radix_enabled)
radix_off_names = ",".join(cfg.name for cfg in WORKLOADS if not cfg.radix_enabled)
bench_on_cmd = [
    sys.executable,
    "scripts/run_bench.py",
    "--configs",
    radix_on_names,
    "--base-url",
    CLOCKWORK_URL,
    "--out",
    "results/clockwork",
]
must(bench_on_cmd)
stop_server(clockwork_proc)
# The radix-off traces replay against a server whose prefix cache is disabled,
# appending to the same summary.csv so every workload has exactly one measured row.
clockwork_off_proc = start_server(
    clockwork_cmd + ["--set", "enable_prefix_cache=false"],
    LOGS / "clockwork_server_radix_off.log",
    f"{CLOCKWORK_URL}/health",
    timeout_s=900,
)
bench_off_cmd = [
    sys.executable,
    "scripts/run_bench.py",
    "--configs",
    radix_off_names,
    "--base-url",
    CLOCKWORK_URL,
    "--out",
    "results/clockwork",
]
must(bench_off_cmd)
stop_server(clockwork_off_proc)
CLOCKWORK_BENCH_CMDS = [" ".join(bench_on_cmd), " ".join(bench_off_cmd)]

## 5b. Derived-metric experiments

Two extra measurements on a fresh clockwork server. First, the same agent workload runs
twice, a cold pass on an empty prefix cache and an identical warm replay, so the TTFT
delta isolates the prefix cache effect on identical requests under an identical
schedule. Second, three agent workloads rerun with fixed-length generation
(ignore_eos, max_tokens 64) on clockwork here and on vLLM in its section below;
identical output token counts make session completion latency comparable across
engines. Expected wall clock: 10 to 20 minutes.


In [ ]:
EXPERIMENT_TTFT_WL = "agent_p1536_t6to12_pois_r2"
FIXEDLEN_WLS = "agent_p1024_t4to8_pois_r2,agent_p1024_t4to8_pois_r4,agent_p1536_t6to12_pois_r2"
exp_proc = start_server(
    clockwork_cmd,
    LOGS / "clockwork_server_experiments.log",
    f"{CLOCKWORK_URL}/health",
    timeout_s=900,
)
for out in ("results/ttft_cold", "results/ttft_warm"):
    must(
        [
            sys.executable,
            "scripts/run_bench.py",
            "--configs",
            EXPERIMENT_TTFT_WL,
            "--base-url",
            CLOCKWORK_URL,
            "--out",
            out,
        ]
    )
must(
    [
        sys.executable,
        "scripts/run_bench.py",
        "--configs",
        FIXEDLEN_WLS,
        "--base-url",
        CLOCKWORK_URL,
        "--out",
        "results/sessions_fixed/clockwork",
        "--max-tokens",
        "64",
        "--ignore-eos",
    ]
)
stop_server(exp_proc)

## 6. vLLM baseline

Installs vLLM into this environment only if pip's resolver would not replace the
installed torch or transformers, otherwise into a fresh venv; the cell prints which happened. When venv cannot
bootstrap pip (Colab system python omits ensurepip), the cell installs python3-venv
via apt or falls back to virtualenv; if vLLM still cannot install, the baseline is
reported as not-run and the rest of the notebook continues. Serves the same model in float16 on its default settings, printed verbatim from
the server's own startup log, then runs the same scripts/run_bench.py workloads against
its port. Expected wall clock: 30 to 90 minutes including the install.

The benchmark loop runs one run_bench invocation per workload and snapshots the
Prometheus /metrics prefix cache counters of vLLM around each one, so the per
workload prefix hit rate of vLLM is measured from counter deltas
(results/vllm/hitrates.csv, folded into its summary.csv). The clockwork config
admits prompts up to max_num_batched_tokens 4096, so all 21 workloads produce data.


In [ ]:
def make_env(path):
    if sh([sys.executable, "-m", "venv", str(path)])[0] == 0:
        return True
    # colab's debian python omits ensurepip, so venv cannot bootstrap pip
    ver = f"python{sys.version_info.major}.{sys.version_info.minor}-venv"
    sh(["apt-get", "update", "-qq"])
    sh(["apt-get", "install", "-qq", "-y", ver, "python3-venv"])
    if sh([sys.executable, "-m", "venv", str(path)])[0] == 0:
        return True
    if sh([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])[0] == 0:
        return sh([sys.executable, "-m", "virtualenv", "-q", str(path)])[0] == 0
    return False


vllm_python = None
rc, dry_lines = sh([sys.executable, "-m", "pip", "install", "--dry-run", "vllm"], echo=False)
would = next((line for line in dry_lines if line.startswith("Would install")), "")
conflict = re.search(r"\b(torch|transformers)-\d", would)
ok = rc == 0 and not conflict
if ok and sh([sys.executable, "-m", "pip", "install", "-q", "vllm"])[0] == 0:
    vllm_python = sys.executable
    print("VLLM: pins compatible, installed in the current environment")
if vllm_python is None:
    if rc != 0:
        reason = "pip resolution failed"
    elif conflict:
        reason = f"pip would replace {conflict.group(1)}"
    else:
        reason = "direct install failed"
    env_dir = BASE / "venv_vllm"
    if make_env(env_dir):
        candidate = str(env_dir / "bin" / "python")
        if sh([candidate, "-m", "pip", "install", "-q", "vllm"])[0] == 0:
            vllm_python = candidate
            print(f"VLLM: {reason}, installed in a fresh venv")
if vllm_python is None:
    print("VLLM: not-run (install failed; see output above)")

In [ ]:
VLLM_BENCH_CMD = None
if vllm_python is None:
    print("VLLM: not-run, baseline skipped")
else:
    VLLM_URL = "http://127.0.0.1:8100"
    vllm_bin = Path(vllm_python).parent / "vllm"
    vllm_cmd = [str(vllm_bin), "serve", MODEL]
    if not vllm_bin.exists():
        vllm_cmd = [vllm_python, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL]
    vllm_cmd += ["--dtype", "float16", "--port", "8100"]
    health = f"{VLLM_URL}/health"
    vllm_proc = start_server(vllm_cmd, LOGS / "vllm_server.log", health, timeout_s=1800)
    print("vllm settings, verbatim from the server log:")
    log_lines = (LOGS / "vllm_server.log").read_text(errors="replace").splitlines()
    shown = [line for line in log_lines if "Namespace(" in line or "EngineArgs(" in line]
    for line in shown or log_lines[:30]:
        print(line)
    import csv
    import urllib.request

    from clockwork.bench.configs import WORKLOADS

    def scrape_prefix_counters(base):
        # vllm v1 exports vllm:prefix_cache_queries and vllm:prefix_cache_hits
        # token counters at /metrics; sum across label sets, tolerate absence.
        try:
            text = urllib.request.urlopen(base + "/metrics", timeout=10).read().decode()
        except Exception:
            return None
        hits, queries, found = 0.0, 0.0, False
        for line in text.splitlines():
            if not line.startswith("vllm:") or "prefix_cache" not in line:
                continue
            name = line.split("{")[0].split(" ")[0]
            try:
                value = float(line.rsplit(" ", 1)[1])
            except ValueError:
                continue
            if "hit" in name:
                hits, found = hits + value, True
            elif "quer" in name:
                queries, found = queries + value, True
        return (hits, queries) if found else None

    hit_rows = []
    for wl in WORKLOADS:
        before = scrape_prefix_counters(VLLM_URL)
        must(
            [
                sys.executable,
                "scripts/run_bench.py",
                "--configs",
                wl.name,
                "--base-url",
                VLLM_URL,
                "--out",
                "results/vllm",
            ]
        )
        after = scrape_prefix_counters(VLLM_URL)
        if before is None or after is None:
            hit_rows.append((wl.name, "", "", ""))
            continue
        dh, dq = after[0] - before[0], after[1] - before[1]
        hit_rows.append((wl.name, dh, dq, dh / dq if dq > 0 else ""))
    must(
        [
            sys.executable,
            "scripts/run_bench.py",
            "--configs",
            FIXEDLEN_WLS,
            "--base-url",
            VLLM_URL,
            "--out",
            "results/sessions_fixed/vllm",
            "--max-tokens",
            "64",
            "--ignore-eos",
        ]
    )
    stop_server(vllm_proc)
    with (REPO / "results" / "vllm" / "hitrates.csv").open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["workload", "prefix_hit_tokens", "prefix_query_tokens", "hit_rate"])
        writer.writerows(hit_rows)
    # Each run_bench invocation rewrites summary.csv with only its own workload,
    # so rebuild the full summary from the per-request CSVs, then fold in the
    # scraped hit rates so every downstream table and figure carries them.
    (REPO / "results" / "vllm" / "summary.csv").unlink(missing_ok=True)
    must([sys.executable, "scripts/collect_results.py", "--out", "results/vllm", "--no-figures"])
    rates = {name: rate for name, _, _, rate in hit_rows if rate != ""}
    summary_path = REPO / "results" / "vllm" / "summary.csv"
    rows = list(csv.DictReader(summary_path.open()))
    for row in rows:
        if row["workload"] in rates:
            row["hit_rate"] = f"{rates[row['workload']]:.4f}"
    with summary_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    scraped = sum(1 for r in hit_rows if r[3] != "")
    print(f"vllm hit rates scraped for {scraped} of {len(hit_rows)} workloads")
    VLLM_BENCH_CMD = (
        "scripts/run_bench.py --configs <name> --base-url " + VLLM_URL + " --out results/vllm"
        " per workload, hit rate from /metrics prefix counter deltas"
    )

In [ ]:
import csv


def _ttfts(path):
    return sorted(
        float(r["ttft_ms"]) for r in csv.DictReader(path.open()) if r["ttft_ms"] and not r["error"]
    )


def _pct(vals, p):
    i = max(0, int(round(p / 100 * len(vals) + 0.5)) - 1)
    return vals[min(i, len(vals) - 1)]


derived = []
cold = _ttfts(REPO / "results" / "ttft_cold" / f"{EXPERIMENT_TTFT_WL}.csv")
warm = _ttfts(REPO / "results" / "ttft_warm" / f"{EXPERIMENT_TTFT_WL}.csv")
if cold and warm:
    for p in (50, 99):
        cut = (1 - _pct(warm, p) / _pct(cold, p)) * 100
        derived += [
            (f"ttft_p{p}_cold_ms", f"{_pct(cold, p):.1f}"),
            (f"ttft_p{p}_warm_ms", f"{_pct(warm, p):.1f}"),
            (f"ttft_p{p}_cut_pct", f"{cut:.1f}"),
        ]
        print(
            f"prefix cache ttft p{p}: cold {_pct(cold, p):.0f} ms, "
            f"warm {_pct(warm, p):.0f} ms, cut {cut:.0f}%"
        )


def _sessions(path):
    lo, hi, bad = {}, {}, 0
    for r in csv.DictReader(path.open()):
        if r["error"]:
            bad += 1
            continue
        k = r["session_id"]
        lo[k] = min(lo.get(k, 1e18), float(r["arrival_s"]))
        hi[k] = max(hi.get(k, 0.0), float(r["end_s"]))
    return {k: hi[k] - lo[k] for k in lo}, bad


totals = [0.0, 0.0, 0]
for name in FIXEDLEN_WLS.split(","):
    cpath = REPO / "results" / "sessions_fixed" / "clockwork" / f"{name}.csv"
    vpath = REPO / "results" / "sessions_fixed" / "vllm" / f"{name}.csv"
    if not (cpath.exists() and vpath.exists()):
        print(f"{name}: fixed-length pair incomplete, skipped")
        continue
    ct, cbad = _sessions(cpath)
    vt, vbad = _sessions(vpath)
    if cbad or vbad:
        print(f"{name}: {cbad} clockwork and {vbad} vllm errored requests excluded")
    common = sorted(set(ct) & set(vt))
    if not common:
        continue
    cm = sum(ct[k] for k in common) / len(common)
    vm = sum(vt[k] for k in common) / len(common)
    totals[0] += sum(ct[k] for k in common)
    totals[1] += sum(vt[k] for k in common)
    totals[2] += len(common)
    print(
        f"{name}: mean session clockwork {cm:.2f}s vllm {vm:.2f}s, "
        f"change {(1 - cm / vm) * 100:+.0f}%"
    )
    derived += [
        (f"session_mean_s_clockwork_{name}", f"{cm:.3f}"),
        (f"session_mean_s_vllm_{name}", f"{vm:.3f}"),
    ]
if totals[2]:
    pooled = (1 - totals[0] / totals[1]) * 100
    word = "faster" if pooled > 0 else "slower"
    print(f"fixed-length sessions pooled over {totals[2]}: clockwork {word} by {abs(pooled):.0f}%")
    derived.append(("session_pooled_change_pct", f"{pooled:.1f}"))
with (REPO / "results" / "derived_metrics.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerows(derived)
print("wrote results/derived_metrics.csv")

## 7. SGLang baseline

Attempts the same install and serve flow for SGLang. Any failure prints
SGLANG: not-run with the reason, and the run continues; SGLang depends on kernels that
may not support the T4's sm75. Expected wall clock: 2 to 30 minutes.

In [ ]:
SGLANG_STATUS = "not-run"
SGLANG_URL = "http://127.0.0.1:8200"
sglang_proc = None
try:
    rc, dry_lines = sh(
        [sys.executable, "-m", "pip", "install", "--dry-run", "sglang[all]"], echo=False
    )
    would = next((line for line in dry_lines if line.startswith("Would install")), "")
    conflict = re.search(r"\b(torch|transformers)-\d", would)
    if rc == 0 and not conflict:
        must([sys.executable, "-m", "pip", "install", "-q", "sglang[all]"])
        sglang_python = sys.executable
        print("SGLANG: installed in the current environment")
    else:
        if not make_env(BASE / "venv_sglang"):
            raise RuntimeError("could not create a venv for sglang")
        sglang_python = str(BASE / "venv_sglang" / "bin" / "python")
        must([sglang_python, "-m", "pip", "install", "-q", "sglang[all]"])
        print("SGLANG: installed in a fresh venv")
    sglang_cmd = [
        sglang_python,
        "-m",
        "sglang.launch_server",
        "--model-path",
        MODEL,
        "--dtype",
        "float16",
        "--port",
        "8200",
    ]
    sglang_proc = start_server(
        sglang_cmd, LOGS / "sglang_server.log", f"{SGLANG_URL}/health", timeout_s=1800
    )
    sglang_bench_cmd = [
        sys.executable,
        "scripts/run_bench.py",
        "--configs",
        "all",
        "--base-url",
        SGLANG_URL,
        "--out",
        "results/sglang",
    ]
    must(sglang_bench_cmd)
    stop_server(sglang_proc)
    SGLANG_STATUS = "ran: " + " ".join(sglang_bench_cmd)
    print("SGLANG: ran")
except Exception as exc:
    SGLANG_STATUS = f"not-run: {exc}"
    print(f"SGLANG: not-run ({exc})")
    if sglang_proc is not None and sglang_proc.poll() is None:
        sglang_proc.kill()

## 8. Results

Renders figures from every summary.csv through the repo's plotting entry point (clockwork
figures at docs/figures/, baselines in per-engine subdirectories), runs
scripts/collect_results.py on each engine's result directory for the markdown tables,
writes the GPU environment beside the CSVs, and zips results/ and docs/figures/ under
/content. Expected wall clock: under 2 minutes.

In [ ]:
from clockwork.bench.plots import plot_all

fig_base = REPO / "docs" / "figures"
for engine in ("clockwork", "vllm", "sglang"):
    summary = REPO / "results" / engine / "summary.csv"
    if not summary.exists():
        print(f"{engine}: no summary.csv, no figures")
        continue
    fig_dir = fig_base if engine == "clockwork" else fig_base / engine
    for figure in plot_all(summary, fig_dir):
        print("figure:", figure)

In [ ]:
env_path = REPO / "results" / "gpu_env.json"
env_path.write_text(
    json.dumps(
        {
            "gpu_name": GPU_NAME,
            "gpu_memory_gb": round(gpu_props.total_memory / 1e9, 2),
            "compute_capability": f"sm{gpu_props.major}{gpu_props.minor}",
            "cuda": torch.version.cuda,
        },
        indent=2,
    )
    + "\n"
)
print("environment:", env_path)
for engine in ("clockwork", "vllm", "sglang"):
    out_dir = REPO / "results" / engine
    if not (out_dir / "summary.csv").exists():
        print(f"{engine}: no summary.csv, no table")
        continue
    print(f"{engine} results:")
    must([sys.executable, "scripts/collect_results.py", "--out", str(out_dir), "--no-figures"])
zip_path = BASE / "clockwork_results_t4.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for tree in (REPO / "results", REPO / "docs" / "figures"):
        for file in sorted(p for p in tree.rglob("*") if p.is_file()):
            zf.write(file, file.relative_to(REPO))
print("results zip:", zip_path)

### Resume numbers

Prints rows formatted for private/RESUME_NUMBERS.md. Every value is read back from files
written earlier in this run and every row names the GPU.

In [ ]:
def read_summary(engine):
    path = REPO / "results" / engine / "summary.csv"
    if not path.exists():
        return []
    with path.open(newline="") as fh:
        return list(csv.DictReader(fh))


def fnum(row, key):
    value = row.get(key, "")
    return float(value) if value not in ("", None) else float("nan")


rows = []
for label, cmd, took in GATE_RESULTS:
    rows.append((f"{label} ({GPU_NAME})", f"PASS, {took:.0f}s", cmd))
best_micro = max(MICRO_ROWS, key=lambda row: row["speedup"])
rows.append(
    (
        f"triton vs torch paged decode ({GPU_NAME})",
        f"winner {BACKEND_WINNER}; best speedup {best_micro['speedup']:.2f}x "
        f"at batch {best_micro['batch']}, ctx {best_micro['ctx_len']}",
        "notebooks/bench_t4.ipynb microbench cell; results/microbench_decode.csv",
    )
)
clock = read_summary("clockwork")
by_name = {row["workload"]: row for row in clock}
if clock:
    peak = max(clock, key=lambda row: fnum(row, "output_tok_s"))
    rows.append(
        (
            f"clockwork peak output tok/s ({GPU_NAME})",
            f"{fnum(peak, 'output_tok_s'):.1f} on {peak['workload']}",
            CLOCKWORK_BENCH_CMDS[0],
        )
    )
    mid = by_name.get("sharegpt_r4")
    if mid:
        rows.append(
            (
                f"clockwork latency on sharegpt_r4 ({GPU_NAME})",
                f"ttft p50 {fnum(mid, 'ttft_p50_ms'):.1f} ms, "
                f"p99 {fnum(mid, 'ttft_p99_ms'):.1f} ms, "
                f"itl p50 {fnum(mid, 'itl_p50_ms'):.1f} ms",
                CLOCKWORK_BENCH_CMDS[0],
            )
        )
    for rate in ("2", "8"):
        on = by_name.get(f"ablation_p1536_radix_on_r{rate}")
        off = by_name.get(f"ablation_p1536_radix_off_r{rate}")
        if on and off:
            rows.append(
                (
                    f"radix ablation at {rate} req/s ({GPU_NAME})",
                    f"ttft p50 {fnum(on, 'ttft_p50_ms'):.1f} ms on vs "
                    f"{fnum(off, 'ttft_p50_ms'):.1f} ms off; "
                    f"hit rate {fnum(on, 'hit_rate'):.2f}",
                    "; ".join(CLOCKWORK_BENCH_CMDS),
                )
            )
vllm_summary = read_summary("vllm")
vllm_by_name = {row["workload"]: row for row in vllm_summary}
common = sorted(set(by_name) & set(vllm_by_name))
if common:
    ours = sum(fnum(by_name[name], "output_tok_s") for name in common) / len(common)
    theirs = sum(fnum(vllm_by_name[name], "output_tok_s") for name in common) / len(common)
    rows.append(
        (
            f"clockwork vs vllm mean output tok/s, {len(common)} shared workloads ({GPU_NAME})",
            f"{ours:.1f} vs {theirs:.1f}",
            VLLM_BENCH_CMD,
        )
    )
rows.append((f"sglang baseline ({GPU_NAME})", SGLANG_STATUS, "notebooks/bench_t4.ipynb section 7"))
print("| value | measurement | source command |")
print("| --- | --- | --- |")
for value, measurement, source in rows:
    print(f"| {value} | {measurement} | {source} |")
print()
print("results zip:", ", ".join(str(p) for p in BASE.glob("clockwork_results_*.zip")))